# VCM_1_0 — Architecture

**Date:** 2026-01-20

This notebook documents the **software architecture** of the Variational Crack Method (VCM) codebase:
module boundaries, key classes/methods, and data flow. Equations are intentionally minimized here;
see the companion **Theory** notebook for numbered derivations and formulas.


## 1. Layered design

VCM is structured into four layers with one-way dependencies:

```
Ugenerator  →  Usolver  →  Uprocessor  →  Uplotter
                          (optional: BEM correction)
```

- **Ugenerator**: geometry/topology generation only  
- **Usolver**: variational assembly + constraints + KKT solve  
- **Uprocessor**: reconstruction of jump, stress/displacement, and fracture metrics (PK, SIF)  
- **Uplotter**: visualization and diagnostics (read-only)


## 2. Terminology

In this codebase, `crack_mode` refers to the elastic source structure:

| Code term | Physical term | Interpretation |
|---|---|---|
| `full` | dipolar | Burgers neutrality constraints enforce dipole-like far field |
| `half` | monopolar | net Burgers content allowed; junction DOFs enforce compatibility |


## 3. Canonical solver entry point

The stable public API for solving crack networks is:

- **Module:** `Usolver/parametrization.py`
- **Class:** `DCENetworkStaticV4`
- **Method:** `solve(...)`

This façade orchestrates:
- assembly helpers in `Usolver/build.py`
- constraint builders in `Usolver/constraints.py`
- KKT solution via `solve_kkt_lsq` in `Usolver/KKT.py`


## 4. Solver pipeline and call graph

`DCENetworkStaticV4.solve()` executes:

1. Validate options; map Chebyshev aliases to `collocation_mode`
2. Compute vertex degree map: `vertex_degrees(network)`
3. Build crack polylines  
   - `build_polylines_full(network)` for `crack_mode='full'`  
   - `build_polylines_half_branches(network, deg)` for `crack_mode='half'`
4. Discretize polylines: `discretize_polylines(...)` → `poly_panels`
5. Allocate unknowns: `allocate_unknowns(...)` → `(offsets, junction_dof, nunk)`
6. Assemble operator: `assemble_operator(...)` → `(K, rhs)`
7. Build constraints  
   - `build_constraints_full(...)`  
   - `build_constraints_half_option_a(...)`
8. Solve KKT system: `solve_kkt_lsq(K, rhs, C, ridge)` → `q`
9. Package solution dictionary `sol`

### Call graph

```
DCENetworkStaticV4.solve
  ├─ vertex_degrees
  ├─ build_polylines_full / build_polylines_half_branches
  ├─ discretize_polylines
  ├─ allocate_unknowns
  ├─ assemble_operator
  ├─ build_constraints_full / build_constraints_half_option_a
  ├─ solve_kkt_lsq
  └─ sol dict
```


## 5. Extension guide (where to change what)

- **New discretization / basis:** implement in `Usolver/build.py`; ensure mid-panel frames for reconstruction  
- **New constraints / junction option:** add builder in `Usolver/constraints.py`  
- **New fracture metric:** add module under `Uprocessor` (stress- or energy-based)  
- **Crack growth law:** add `Uprocessor/growth.py` consuming $K_I$, $K_{II}$, or PK forces  
- **New plot:** extend `Uplotter/core.py` or `Uplotter/plot_PK.py` (read-only)


## 6. Notebook bootstrap (recommended)

In [ ]:
import os, sys
from pathlib import Path

cwd = Path(os.getcwd()).resolve()
root = next((p for p in [cwd] + list(cwd.parents) if (p / "preamble.py").exists()), None)
if root is None:
    raise RuntimeError("preamble.py not found in current directory or any parent")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Repo root:", root)
